<style>
/* ── 전체 레이아웃 — 중앙 정렬 + 양쪽 여백 ── */
body {
  max-width: 860px;
  margin: 0 auto;
  padding: 0 32px;
  box-sizing: border-box;
}

/* ── 전체 폰트 ── */
body, .jp-RenderedMarkdown {
  font-family: 'Noto Sans KR', 'Apple SD Gothic Neo', 'Malgun Gothic', sans-serif;
  color: #222;
  line-height: 1.7;
}

/* ── 타이틀 헤더 블록 ── */
.jp-RenderedMarkdown h1:first-of-type,
h1:first-of-type {
  background: linear-gradient(135deg, #105F68 0%, #3A9295 100%);
  color: white !important;
  padding: 32px 36px 28px;
  border-radius: 8px;
  margin: 0 0 24px 0;
  font-size: 1.55em;
  letter-spacing: -0.3px;
}

/* ── 섹션 제목 ── */
h2 {
  color: #105F68;
  border-bottom: 2px solid #63C1BB;
  padding-bottom: 6px;
  margin-top: 28px;
}

/* ── 표 좌측 정렬 & 스타일 ── */
table {
  margin-left: 0 !important;
  margin-right: auto !important;
  border-collapse: collapse;
  font-size: 0.95em;
}
table th, table td {
  text-align: left !important;
  padding: 7px 24px 7px 0;
}
table thead tr {
  border-bottom: 2px solid #63C1BB;
}
table tbody tr + tr {
  border-top: 1px solid #eee;
}

/* ── 블록쿼트 → 콜아웃 박스 ── */
blockquote {
  border-left: 4px solid #63C1BB;
  background: #F0FAFA;
  margin: 12px 0;
  padding: 12px 18px;
  border-radius: 0 6px 6px 0;
  color: #333;
  font-style: normal;
}

/* ── 결론 강조 ── */
hr + p strong, p strong {
  font-size: 1.02em;
}

/* ── 차트·데이터프레임 가로 스크롤 (모바일 공통) ── */
.jp-OutputArea-output,
.jp-Cell-outputWrapper {
  overflow-x: auto;
}

/* ── 모바일 반응형 ── */
@media (max-width: 768px) {
  body {
    font-size: 14px;
    padding: 0 16px;
  }

  h1:first-of-type {
    font-size: 1.15em !important;
    padding: 20px 18px !important;
    border-radius: 6px !important;
  }

  h2 {
    font-size: 1.05em;
  }

  blockquote {
    padding: 10px 14px;
    font-size: 0.92em;
  }

  .vega-embed {
    overflow-x: auto;
    display: block;
    -webkit-overflow-scrolling: touch;
  }

  .jp-RenderedHTMLCommon {
    overflow-x: auto;
    -webkit-overflow-scrolling: touch;
  }

  .jp-RenderedMarkdown table {
    display: block;
    overflow-x: auto;
    white-space: nowrap;
  }
}
</style>

# 수도권 주말 매치팀 구인 게시글 — 지역 빈도 분석 리포트

## 개요
다음 카페 '서경 축구클럽'에서 주말 매치팀 구인 게시글을 수집·분석하여,  
수도권 내 지역별 경기 수요 분포를 시각화합니다.

## 데이터 수집 정보
| 항목 | 내용 |
|:------|:------|
| 수집 소스 | 다음 카페 (`cafe.daum.net/skfootball`) |
| 수집 게시판 | 서울북부 · 서울남부 · 경기북부 · 경기남부 · 인천부천 주말축구 |
| 수집 기간 | 2026-05-08 ~ 2026-05-21 (2주) |
| 장소 매핑 | 게시글 제목에서 키워드 추출 → 시도 / 시군구 매핑 |

In [19]:
import importlib
import altair as alt
import json
import pandas as pd
from pathlib import Path

import data_loader as _dl_mod, charts as _charts_mod
importlib.reload(_dl_mod)
importlib.reload(_charts_mod)

from data_loader import load_data, ZONE_MAP
from charts import (
    sido_bar_chart, sigungu_bar_chart, interactive_sido_sigungu_chart,
    board_pie_chart, zone_bar_chart, interactive_zone_sigungu_chart, AXIS_CONFIG,
)

df = load_data()
# print(df.shape)
# df.head()

In [20]:
data_dir = Path("data")

with open(data_dir / "raw_posts.json", encoding="utf-8") as f:
    raw_posts = json.load(f)
    raw_count = len(raw_posts)

with open(data_dir / "raw_posts_dedup.json", encoding="utf-8") as f:
    dedup_count = len(json.load(f))

df_raw = pd.DataFrame(raw_posts)

mapped   = df["venue"].notna().sum()
unmapped = df["venue"].isna().sum()
date_min = df["created"].min().date()
date_max = df["created"].max().date()

# print("=" * 40)
# print("  데이터 기본 정보")
# print("=" * 40)
# print(f"원본 수집 게시글    : {raw_count:,}개")
# print(f"중복 제거 후        : {dedup_count:,}개  (제거: {raw_count - dedup_count:,}개)")
# print(f"분석 대상 (처리 후) : {len(df):,}개")
# print()
# print(f"게시글 작성 기간    : {date_min} ~ {date_max}")
# print()
# print("[ 게시판별 게시글 수 ]")
# print(df["board"].value_counts().to_string())
# print()
# print("[ 장소 매핑률 ]")
# print(f"매핑 성공  : {mapped:,}개 ({mapped / len(df) * 100:.1f}%)")
# print(f"매핑 실패  : {unmapped:,}개 ({unmapped / len(df) * 100:.1f}%)")

In [21]:
_board_counts = df_raw["board"].value_counts().reset_index()
_board_counts.columns = ["board", "count"]

pie = board_pie_chart(
    df_raw,
    title=f"게시판별 게시글 수 (전체 {raw_count:,}개)"
)

# _bar_base = alt.Chart(_board_counts)

# bar = _bar_base.mark_bar().encode(
#     x=alt.X("count:Q", title="게시글 수"),
#     y=alt.Y("board:N", sort="-x", title=None,
#             scale=alt.Scale(paddingOuter=0.6)),
#     color=alt.condition(
#         alt.datum.board == "경기북부 주말축구",
#         alt.value("#F78535"),
#         alt.value("#63C1BB")
#     ),
#     tooltip=["board:N", "count:Q"]
# )

# bar_text = _bar_base.mark_text(align="left", dx=4, fontSize=11).encode(
#     x=alt.X("count:Q"),
#     y=alt.Y("board:N", sort="-x", scale=alt.Scale(paddingOuter=0.4)),
#     text=alt.Text("count:Q", format=",d"),
# )

# bar_combined = (bar + bar_text).properties(
#     width=300,
#     height=200,
#     title="게시판별 게시글 수"
# )

# alt.hconcat(pie, bar_combined).configure_axis(**AXIS_CONFIG).configure_view(stroke=None)
pie

alt.LayerChart(...)

> 기간동안 올라온 게시물 중 '경기북부' 게시물은 약 10% 정도

In [22]:
_day_en = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}

df_daily = (
    df.groupby(df['created'].dt.strftime('%Y-%m-%d'))
    .size()
    .reset_index(name='count')
    .rename(columns={'created': 'date'})
)
df_daily['label'] = df_daily['date'].apply(
    lambda s: (
        lambda t: f"{t.month}/{t.day:02d}({_day_en[t.weekday()]})"
    )(pd.Timestamp(s))
)

tick_labels = list(df_daily['label'][::2])

alt.Chart(df_daily).mark_line(color="#63C1BB", interpolate="monotone", point=alt.OverlayMarkDef(color="#63C1BB")).encode(
    x=alt.X("label:O", sort=list(df_daily['label']), title="날짜",
            axis=alt.Axis(labelAngle=0, values=tick_labels)),
    y=alt.Y("count:Q", title="게시글 수"),
    tooltip=[
        alt.Tooltip("label:N", title="날짜"),
        alt.Tooltip("count:Q", title="게시글 수"),
    ]
).properties(
    title="날짜별 게시글 수",
    width=600,
    height=250,
).configure_axis(**AXIS_CONFIG).configure_view(stroke=None)

alt.Chart(...)

> 게시글이 가장 활발하게 올라오는 시점은 매주 월요일

In [23]:
step_order = ["원본 수집", "중복 제거 후", "매핑 완료"]

funnel_df = pd.DataFrame([
    {"단계": "원본 수집",    "count": raw_count,   "label": f"{raw_count:,}개"},
    {"단계": "중복 제거 후", "count": dedup_count,  "label": f"{dedup_count:,}개"},
    {"단계": "매핑 완료",    "count": mapped,       "label": f"{mapped:,}개"},
])
funnel_df["left"]  = -(funnel_df["count"] / 2)
funnel_df["right"] =  (funnel_df["count"] / 2)

note_df = pd.DataFrame([
    {"단계": "중복 제거 후", "note": f"▼ 중복 {raw_count - dedup_count:,}개 제거"},
    {"단계": "매핑 완료",    "note": f"▼ 미매핑 {unmapped:,}개 제외"},
])

bars = alt.Chart(funnel_df).mark_bar(height=44).encode(
    x=alt.X("left:Q", axis=None, scale=alt.Scale(domain=[-700, 870])),
    x2="right:Q",
    y=alt.Y("단계:N", sort=step_order, title=None),
    color=alt.Color(
        "단계:N", legend=None,
        scale=alt.Scale(domain=step_order, range=["#D9D9D9", "#8ECECA", "#63C1BB"])
    ),
    tooltip=[
        alt.Tooltip("단계:N", title="단계"),
        alt.Tooltip("count:Q", title="게시글 수", format=",d"),
    ]
)

center_text = alt.Chart(funnel_df).mark_text(
    fontSize=14, fontWeight="bold", color="white"
).encode(
    x=alt.datum(0),
    y=alt.Y("단계:N", sort=step_order),
    text="label:N"
)

side_notes = alt.Chart(note_df).mark_text(
    fontSize=11, color="#999", align="left", dx=8
).encode(
    x=alt.datum(630),
    y=alt.Y("단계:N", sort=step_order),
    text="note:N"
)

(bars + center_text + side_notes).properties(
    title="분석에 사용한 데이터 수",
    width=600,
    height=200,
).configure_axis(**AXIS_CONFIG).configure_view(stroke=None)

alt.LayerChart(...)

> 총 1,248개 데이터 중 아래 차트에서 사용되는 데이터는 461개

In [24]:
# !pip install geopandas

In [25]:
import requests
import geopandas as gpd
from shapely.geometry import shape

url = 'https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_municipalities_geo_simple.json'
korea_geojson = requests.get(url).json()

# 통계청 5자리 코드: 서울=11, 인천=23, 경기=31 / 도서 지역 제외
SIDO_CODE = {'11': '서울', '23': '인천', '31': '경기'}
EXCLUDE    = {'옹진군', '강화군'}
features_sudogwon = [
    f for f in korea_geojson['features']
    if f['properties']['code'][:2] in SIDO_CODE
    and f['properties']['name'] not in EXCLUDE
]
sudogwon_geojson = {**korea_geojson, 'features': features_sudogwon}

def get_zone(sigungu, sido):
    if sido == '인천':
        return '경기 인천권'
    if sigungu == '중구':          # 서울 중구 → 서울 도심 (인천 중구는 위에서 처리)
        return '서울 도심'
    zone = ZONE_MAP.get(sigungu)
    if zone is None:
        # "고양시 일산동구" 등 구 단위 분리 → 시 이름으로 fallback
        for key in ZONE_MAP:
            if sigungu.startswith(key):
                return ZONE_MAP[key]
    return zone

rows = []
for f in features_sudogwon:
    sigungu = f['properties']['name']
    sido    = SIDO_CODE[f['properties']['code'][:2]]
    rows.append({'sigungu': sigungu, 'sido': sido, 'zone': get_zone(sigungu, sido),
                 'geometry': shape(f['geometry'])})

sudogwon_gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
# print(f"수도권 시군구: {len(sudogwon_gdf)}개")
# sudogwon_gdf[['sigungu', 'sido', 'zone']].head(10)

In [26]:
# 시군구 geometry를 권역 단위로 병합 (dissolve)
zone_gdf = sudogwon_gdf.dissolve(by='zone').reset_index()[['zone', 'geometry']]

zone_count_df = (
    df.dropna(subset=['zone'])
    .groupby('zone').size()
    .reset_index(name='count')
)
zone_plot = zone_gdf.merge(zone_count_df, on='zone', how='left')
zone_plot['count'] = zone_plot['count'].fillna(0).astype(int)
max_count = int(zone_plot['count'].max())

point_nearest   = alt.selection_point(fields=['zone'], on='pointerover')
point_selection = alt.selection_point(fields=['zone'])

# ── 지도 (권역 단위) ────────────────────────────────────
map_colored = (
    alt.Chart(zone_plot)
    .mark_geoshape(stroke='white', strokeWidth=1.5)
    .encode(
        color=alt.Color(
            'count:Q',
            scale=alt.Scale(range=['#E0F4F3', '#63C1BB'], domain=[0, max_count]),
            legend=None
        ),
        tooltip=[
            alt.Tooltip('zone:N',  title='권역'),
            alt.Tooltip('count:Q', title='게시글 수'),
        ]
    )
    .add_params(point_nearest, point_selection)
)

# 경기 서북 하이라이트 (반투명 주황 오버레이)
map_highlight = (
    alt.Chart(zone_plot[zone_plot['zone'] == '경기 서북'])
    .mark_geoshape(fill='#F78535', stroke='white', strokeWidth=1.5, opacity=0.45)
    .encode(
        tooltip=[
            alt.Tooltip('zone:N',  title='권역'),
            alt.Tooltip('count:Q', title='게시글 수'),
        ]
    )
    .add_params(point_nearest, point_selection)
)

map_hover = (
    alt.Chart(zone_plot)
    .mark_geoshape(fill='none', stroke='white', strokeWidth=3)
    .transform_filter(point_nearest)
)
map_selected = (
    alt.Chart(zone_plot)
    .mark_geoshape(fill='none', stroke='#444', strokeWidth=2.5)
    .transform_filter(point_selection)
)
colored_map = alt.layer(map_colored, map_highlight, map_hover, map_selected)

# ── 바차트 (선택된 권역의 시군구별) ──────────────────────
_HIGHLIGHT_SIGUNGU = ['고양시', '파주시', '김포시']

bar_chart = (
    alt.Chart(df.dropna(subset=['zone', 'sigungu']))
    .transform_filter(point_selection)
    .encode(
        x=alt.X('count():Q', title='게시글 수'),
        y=alt.Y('sigungu:N', sort='-x', title=None),
        color=alt.condition(
            alt.FieldOneOfPredicate(field='sigungu', oneOf=_HIGHLIGHT_SIGUNGU),
            alt.value('#F78535'),
            alt.value('#63C1BB')
        ),
        tooltip=['sigungu:N', 'count():Q'],
    )
    .mark_bar()
)

alt.hconcat(
    colored_map.properties(width=400, height=440),
    bar_chart.properties(
        width=280,
        height=440,
        title=alt.Title(
            text='권역별 시군구 게시글 수',
            subtitle='왼쪽 지도를 클릭하여 권역별로 확인할 수 있습니다.'
        )
    )
).configure_axis(**AXIS_CONFIG).configure_view(stroke=None)

alt.HConcatChart(...)

> 타겟으로 하려고하는 '경기 서북(고양, 김포, 파주)'에는 게시물이 14개 있음
>
> (클릭을 통해 탐색가능 - 더블 클릭 시 전체보기)

In [27]:
# 속성 확인용 (디버깅)
# print([f['properties'] for f in korea_geojson['features'][:3]])/

In [28]:
from IPython.display import display
_sigungu = "고양시"
display(
    df[df["sigungu"] == _sigungu][["title", "board", "created", "venue", "sigungu"]]
    .reset_index(drop=True)
)

# 인터랙티브 위젯 (Jupyter 전용 — HTML 내보내기 시 주석 처리)
# import ipywidgets as widgets
# sigungu_list = ["(전체)"] + sorted(df["sigungu"].dropna().unique())
# @widgets.interact(
#     시군구=widgets.Dropdown(options=sigungu_list, value="고양시"),
#     미매핑만=widgets.Checkbox(value=False, description="venue 없는 행만")
# )
# def show(시군구, 미매핑만):
#     result = df if 시군구 == "(전체)" else df[df["sigungu"] == 시군구]
#     if 미매핑만:
#         result = result[result["venue"].isna()]
#     display(result[["title", "board", "created", "venue", "sigungu"]].reset_index(drop=True))

,title,board,created,venue,sigungu
0,[초청] 5/23 10-13토요일 덕은동한강둔치 인조구장,서울북부 주말축구,2026-05-18,덕은동한강둔치구장,고양시
1,5/16 토 오전 10-13시 한강둔치축구장(덕은동) 선출 없는 하하팀만 모십니다,서울북부 주말축구,2026-05-14,덕은동한강둔치구장,고양시
2,26. 5. 16. 토 10-12시 항공대학교 운동장,서울북부 주말축구,2026-05-13,항공대학교운동장,고양시
3,고양시 5/16 토 7-10시 대화체육공원,서울북부 주말축구,2026-05-13,대화체육공원,고양시
4,5/16 토 7-10시 대화체육공원,서울북부 주말축구,2026-05-12,대화체육공원,고양시
5,고양시 6/6 토 7-10시 지영구장,경기북부 주말축구,2026-05-21,지영구장,고양시
6,고양시 5/30 토 7-10시 대화체육공원,경기북부 주말축구,2026-05-21,대화체육공원,고양시
7,2026.05.24 경기도 고양시 대화구장 10시~13시,경기북부 주말축구,2026-05-20,대화구장,고양시
8,이번주 16일 토요일 7-10시 대화구장,경기북부 주말축구,2026-05-14,대화구장,고양시
9,5.16 토요일 07-10 충장체육공원 초청합니다(경기 고양) 급급!!!!,경기북부 주말축구,2026-05-13,충장체육공원,고양시


> 중복제거가 제대로 안된 케이스들이 보임

In [29]:
unmapped_df = df[df["venue"].isna()]

alt.Chart(unmapped_df).mark_bar().encode(
    x=alt.X("count():Q", title="게시글 수"),
    y=alt.Y("board:N", sort="-x", title="게시판"),
    color=alt.condition(
        alt.datum.board == "경기북부 주말축구",
        alt.value("#F78535"),
        alt.value("#63C1BB")
    ),
    tooltip=["board:N", "count():Q"]
).properties(
    title="매핑 안 된 게시글 — 게시판별 현황",
    width=500,
    height=200,
).configure_axis(**AXIS_CONFIG).configure_view(stroke=None)

alt.Chart(...)

In [30]:
df[df["venue"].isna()][["title", "board", "created"]].reset_index(drop=True).head(10)

,title,board,created
0,5월 30일 오전 6-8시 과기대 초청드립니다!(교환 우선),서울북부 주말축구,2026-05-21
1,이번주 상대팀 급구합니다!!!,서울북부 주말축구,2026-05-21
2,"23일 토요일 오전 7시-9시 서강대(마포, 신촌) 무료 초청합니다",서울북부 주말축구,2026-05-21
3,[망원유수지_무료 초청] 6월 6일부터 매월 첫주 토요일 13~16시 매치팀 구합니다,서울북부 주말축구,2026-05-21
4,"(무료) 5/24, 31 일요일 12-3시 신정고등학교 초청합니다",서울북부 주말축구,2026-05-21
5,5월 30일 토요일 노원구 오전 06시 - 08시 초청합니다.(6월 13일 교환 우선),서울북부 주말축구,2026-05-21
6,(무료) 5/24 일요일 12-3시 신정고등학교 초청합니다,서울북부 주말축구,2026-05-21
7,"5월 23일 토요일 오전 7시-9시 서강대(마포, 신촌) 무료 초청합니다",서울북부 주말축구,2026-05-20
8,25일 월요일 관악구민1 야간 상대팀 구합니다.(대체휴무일),서울북부 주말축구,2026-05-20
9,[서울북부_망원유수지_무료 초청] 6월 6일부터 매월 첫주 토요일 13~16시 매치...,서울북부 주말축구,2026-05-20


> 경기장을 매핑 못하는 케이스도 사람이 읽고 수작업으로는 가능해보이는 상황
>
> 그 외에는 제목에 경기장을 명시하지 않은 케이스

---
**결론: 데이터 정제 과정을 다시 거치는게 좋아보인다.**

---